# Adaptive LameR

Run BM25 first. If the mean cosine similarity between the query and the top-k retrieved passages is below a threshold, invoke LameR to augment the query; otherwise keep the BM25 results.

In [14]:
import sys
import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# Make ``src`` importable when running from the ``notebooks/`` directory.
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.retriever import Retriever, create_retriever_callable
from src.agents.lamer import LameRAgent

## Configuration

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

class Config:
    # Data paths (same as test_isolated_lamer.ipynb)
    QUERIES_PATH = PROJECT_ROOT / "notebooks" / "queries" / "topics.ms-marco-dev2.tsv"
    QRELS_PATH = PROJECT_ROOT / "notebooks" / "qrels" / "qrels.ms-marco-dev2.tsv"

    # Evaluation scope
    NUM_QUERIES = 50
    NDCG_K = 50
    RECALL_K = 100

    # Adaptive threshold
    SIMILARITY_THRESHOLD = 0.5  # if mean(query, top-k passages) cosine similarity is below this, call LameR
    TOP_K_FOR_SIMILARITY = 10   # number of top BM25 passages used for the similarity decision

    # Agent hyperparameters
    N_CANDIDATES = 5
    TOP_K_INITIAL = 20
    TOP_K_FINAL = 50

    # Output
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    OUTPUT_CSV = OUTPUT_DIR / "adaptive_lamer_results.csv"

cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
LLM_NAME = "google/gemma-4-E4B-it"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12629.38it/s]


## Data loading helpers

In [16]:
def load_qrels(qrels_path: Path) -> Dict[str, Dict[str, int]]:
    """Load qrels as ``{query_id: {doc_id: relevance_grade}}``."""
    qrels = defaultdict(dict)
    if not qrels_path.exists():
        raise FileNotFoundError(f"Qrels file not found: {qrels_path}")

    with open(qrels_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            query_id, doc_id, grade_str = parts[0].strip(), parts[2].strip(), parts[3].strip()
            try:
                grade = int(grade_str)
            except ValueError:
                continue
            qrels[query_id][doc_id] = grade
    return dict(qrels)


def load_queries(queries_path: Path, num_queries: int = None) -> List[Tuple[str, str]]:
    """Load queries as ``[(query_id, query_text), ...]``."""
    queries = []
    if not queries_path.exists():
        raise FileNotFoundError(f"Queries file not found: {queries_path}")

    with open(queries_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                query_id, query_text = parts[0].strip(), parts[1].strip()
            else:
                query_id, query_text = str(len(queries)), parts[0].strip()
            queries.append((query_id, query_text))
            if num_queries is not None and len(queries) >= num_queries:
                break
    return queries

## Metric helpers

In [17]:
def _dcg(relevances: np.ndarray, k: int) -> float:
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size == 0:
        return 0.0
    positions = np.arange(2, relevances.size + 2)
    return float(np.sum(relevances / np.log2(positions)))


def normalize_doc_id(doc_id: str) -> str:
    """Strip segment suffix (e.g. 'doc#1' -> 'doc') to match qrels format."""
    return doc_id.split("#", 1)[0] if "#" in doc_id else doc_id


def deduplicate_doc_ids(doc_ids: List[str]) -> List[str]:
    """Normalize then deduplicate doc IDs."""
    deduped = []
    seen = set()
    for doc_id in doc_ids:
        normalized = normalize_doc_id(doc_id)
        if normalized not in seen:
            deduped.append(normalized)
            seen.add(normalized)
    return deduped


def compute_ndcg(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 10) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    gains = [qrels.get(doc_id, 0) for doc_id in ranked_docs]
    ideal = sorted((rel for rel in qrels.values() if rel > 0), reverse=True)[:k]
    dcg = _dcg(np.array(gains, dtype=float), k)
    idcg = _dcg(np.array(ideal, dtype=float), k)
    return dcg / idcg if idcg > 0 else 0.0


def compute_recall(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 100) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    relevant = {d for d, r in qrels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(set(ranked_docs) & relevant) / len(relevant)

## Adaptive similarity helper

In [18]:
def mean_query_passage_similarity(
    query_text: str,
    doc_ids: List[str],
    corpus: Dict[str, str],
    embed_model: SentenceTransformer,
    top_k: int = None,
) -> float:
    """
    Embed the query and the first ``top_k`` retrieved passages, then return
    the mean cosine similarity between the query and each passage.
    """
    if top_k is not None:
        doc_ids = doc_ids[:top_k]

    passages = [corpus.get(doc_id, "").strip() for doc_id in doc_ids]
    passages = [p for p in passages if p]

    if not passages:
        return 0.0

    query_embedding = embed_model.encode(query_text, convert_to_numpy=True)
    passage_embeddings = embed_model.encode(passages, convert_to_numpy=True)

    # Cosine similarity = dot product if embeddings are normalized;
    # compute it explicitly to be safe.
    query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
    passage_norms = passage_embeddings / (np.linalg.norm(passage_embeddings, axis=1, keepdims=True) + 1e-10)
    similarities = passage_norms @ query_norm

    return float(np.mean(similarities))

## Initialize retriever, embedding model, and LameR agent

In [19]:
retriever_instance = Retriever(
    endpoint=os.getenv("RETRIEVAL_ENDPOINT"),
    username=os.getenv("MY_USERNAME"),
    password=os.getenv("MY_PASSWORD"),
    index_field="segment",
    top_k=cfg.TOP_K_FINAL,
)
retriever_func = create_retriever_callable(retriever_instance)

lamer_agent = LameRAgent(
    embed_model=EMBED_MODEL,
    n_candidates=cfg.N_CANDIDATES,
    top_k_initial=cfg.TOP_K_INITIAL,
    top_k_final=cfg.TOP_K_FINAL,
    model_name=LLM_NAME,
)

[LameR] LLM client config: base_url=https://hub.nhr.fau.de/api/llmgw/v1, model=google/gemma-4-E4B-it


## Load data

In [20]:
queries = load_queries(cfg.QUERIES_PATH, num_queries=cfg.NUM_QUERIES)
qrels = load_qrels(cfg.QRELS_PATH)

print(f"Loaded {len(queries)} queries.")
print(f"Loaded qrels for {len(qrels)} queries.")

Loaded 50 queries.
Loaded qrels for 5000 queries.


## Run adaptive evaluation

For each query:
1. Retrieve top-k with BM25.
2. Compute mean cosine similarity between query and top-k passages.
3. If similarity < threshold, invoke LameR; otherwise keep BM25 results.
4. Evaluate both BM25 and adaptive result sets.

In [21]:
records = []

for query_id, query_text in queries:
    print(f"[{len(records)+1}/{len(queries)}] Query {query_id}: {query_text[:60]}...")

    # Baseline BM25
    bm25_start = time.time()
    bm25_doc_ids, bm25_scores, bm25_corpus = retriever_func(query_text, cfg.TOP_K_FINAL)
    bm25_elapsed = time.time() - bm25_start

    # Adaptive decision
    decision_start = time.time()
    mean_sim = mean_query_passage_similarity(
        query_text=query_text,
        doc_ids=bm25_doc_ids,
        corpus=bm25_corpus,
        embed_model=EMBED_MODEL,
        top_k=cfg.TOP_K_FOR_SIMILARITY,
    )

    if mean_sim < cfg.SIMILARITY_THRESHOLD:
        # Low similarity: invoke LameR
        effects = lamer_agent.compute_effects({
            "query_text": query_text,
            "retriever": retriever_func,
            "top_k": cfg.TOP_K_FINAL,
        })
        adaptive_doc_ids = effects["new_doc_ids"]
        augmented_query = effects["new_query_text"]
        used_lamer = True
        lamer_cost = effects["cost"]
    else:
        # High similarity: keep BM25 results
        adaptive_doc_ids = bm25_doc_ids
        augmented_query = query_text
        used_lamer = False
        lamer_cost = 0.0

    decision_elapsed = time.time() - decision_start

    qrels_for_query = qrels.get(query_id, {})

    # Metrics
    bm25_ndcg = compute_ndcg(bm25_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    adaptive_ndcg = compute_ndcg(adaptive_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    bm25_recall = compute_recall(bm25_doc_ids, qrels_for_query, k=cfg.RECALL_K)
    adaptive_recall = compute_recall(adaptive_doc_ids, qrels_for_query, k=cfg.RECALL_K)

    records.append({
        "query_id": query_id,
        "query_text": query_text,
        "mean_similarity": mean_sim,
        "used_lamer": used_lamer,
        "augmented_query": augmented_query,
        "bm25_doc_ids": ";".join(bm25_doc_ids),
        "adaptive_doc_ids": ";".join(adaptive_doc_ids),
        "bm25_ndcg": bm25_ndcg,
        "adaptive_ndcg": adaptive_ndcg,
        "ndcg_gain": adaptive_ndcg - bm25_ndcg,
        "bm25_recall": bm25_recall,
        "adaptive_recall": adaptive_recall,
        "recall_gain": adaptive_recall - bm25_recall,
        "bm25_latency_ms": bm25_elapsed * 1000,
        "adaptive_latency_ms": decision_elapsed * 1000,
        "augmented_extra_tokens": len(augmented_query.split()) - len(query_text.split()),
        "lamer_cost": lamer_cost,
    })

df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Saved rankings + metrics to: {cfg.OUTPUT_CSV}")

[1/50] Query 1048579: what is pcnt...


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[LameR] Final candidates (5): ['Pericentrin', 'Panama Canal Nett Tonnage', 'Percutaneous Nephrostomy Tube', 'Pericentrin Gene', 'Purified Carbon Nanotubes']


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[2/50] Query 262156: how long is a college hockey game...


...

[50/50] Query 1051401: what is meant by robust software...


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[LameR] Final candidates (5): ['It refers to software that is reliable and durable, such as in timepieces.', 'It can describe software that is comprehensive and capable, like a suite of tools.', 'It can indicate software that is well-built and dependable, such as in design tools.', 'It suggests software that is powerful and resilient in its operation.', 'It implies software that functions reliably under various conditions.']


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Saved rankings + metrics to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\adaptive_lamer_results.csv


## Summarize

In [22]:
df = pd.DataFrame(records)

print(f"\nSaved per-query results to: {cfg.OUTPUT_CSV}")
print(f"Evaluated queries: {len(df)}")
print(f"Queries where LameR was invoked: {df['used_lamer'].sum()} ({df['used_lamer'].mean():.1%})")
print(f"Queries where at least one retrieval method retrieved a relevant document: {(df['bm25_ndcg'] + df['adaptive_ndcg'] > 0).sum()}")

print("\n=== Overall Averages ===")
print(f"BM25       nDCG@{cfg.NDCG_K}:     {df['bm25_ndcg'].mean():.4f}")
print(f"Adaptive   nDCG@{cfg.NDCG_K}:     {df['adaptive_ndcg'].mean():.4f}")
print(f"Mean nDCG gain:                 {df['ndcg_gain'].mean():+.4f}")
print(f"Win rate (Adaptive > BM25):     {(df['ndcg_gain'] > 0).mean():.1%}")

print(f"\nBM25       Recall@{cfg.RECALL_K}:   {df['bm25_recall'].mean():.4f}")
print(f"Adaptive   Recall@{cfg.RECALL_K}:   {df['adaptive_recall'].mean():.4f}")
print(f"Mean Recall gain:               {df['recall_gain'].mean():+.4f}")

print(f"\nBM25     latency: {df['bm25_latency_ms'].mean():.1f} ms/query")
print(f"Adaptive latency: {df['adaptive_latency_ms'].mean():.1f} ms/query")
print(f"Total LameR cost: {df['lamer_cost'].sum():.0f} candidate calls")

print("\n=== Top 10 nDCG gains (Adaptive vs BM25) ===")
print(df.sort_values("ndcg_gain", ascending=False)[[
    "query_id", "query_text", "mean_similarity", "used_lamer", "ndcg_gain", "bm25_ndcg", "adaptive_ndcg"
]].head(10).to_string(index=False))

print("\n=== Top 10 nDCG losses (Adaptive vs BM25) ===")
print(df.sort_values("ndcg_gain", ascending=True)[[
    "query_id", "query_text", "mean_similarity", "used_lamer", "ndcg_gain", "bm25_ndcg", "adaptive_ndcg"
]].head(10).to_string(index=False))


Saved per-query results to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\adaptive_lamer_results.csv
Evaluated queries: 50
Queries where LameR was invoked: 20 (40.0%)
Queries where at least one retrieval method retrieved a relevant document: 26

=== Overall Averages ===
BM25       nDCG@50:     0.2368
Adaptive   nDCG@50:     0.2807
Mean nDCG gain:                 +0.0440
Win rate (Adaptive > BM25):     12.0%

BM25       Recall@100:   0.4900
Adaptive   Recall@100:   0.4900
Mean Recall gain:               +0.0000

BM25     latency: 1401.1 ms/query
Adaptive latency: 6176.8 ms/query
Total LameR cost: 82 candidate calls

=== Top 10 nDCG gains (Adaptive vs BM25) ===
query_id                                      query_text  mean_similarity  used_lamer  ndcg_gain  bm25_ndcg  adaptive_ndcg
  785665                                 what is postage         0.350631        True   0.729762   0.270238       1.000000
  524574       